# Tüdőrák predikciója vizelet LC–MS metabolomikai profilból

### feature selection és gépi tanulási modellek összehasonlítása

### Ambrus Csaba


## 0. Setup

In [ ]:
# Init gdrive and python environment

from google.colab import drive
drive.mount('/content/drive')

%cd /content

import os
if not os.path.exists("/content/MetabolKD/.git"):
    !git clone https://github.com/csambrus/MetabolKD.git
else:
    %cd /content/MetabolKD
    !git fetch origin
    !git pull origin main

%cd /content/MetabolKD
!pip install -r ./requirements.txt

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = "content/MetabolKD"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
from src.runtime_setup import setup_tensorflow_runtime, setup_notebook_error_logger, set_global_seed

set_global_seed()
setup_notebook_error_logger()
setup_tensorflow_runtime()


## 1. Adatbetöltés


In [ ]:
from src.download_dataset import download_dataset, combine_pos_neg_to_feature_matrix

data = download_dataset()
X, meta = combine_pos_neg_to_feature_matrix(data)

# ML célváltozó
target_col = "class_label"  # Lung cancer vs Control

# Csak címkézett minták
mask = meta[target_col].notna()
X = X.loc[mask].copy()
meta = meta.loc[mask].copy()

# Biztonság: indexek egyezzenek
meta = meta.reindex(X.index)

X_raw = X
y = meta[target_col]

print("X_raw:", X_raw.shape)
print(y.value_counts(dropna=False))

## 2. QC és preprocessing

In [ ]:
from src.metabolomics_qc import qc_sample_table
from src.metabolomics_plotting import plot_qc_bars

qc = qc_sample_table(X_raw)
print(qc.describe())
fig = plot_qc_bars(qc, meta, hue_col="class_label")
fig.savefig("qc_overview_mtbls28.png", dpi=150, bbox_inches="tight")


### Preprocessing

In [ ]:
from src.metabolomics_preprocessing import nmr_style_matrices, preprocess_feature_matrix

blocks = nmr_style_matrices(X_raw, impute_first=True)
for k, df in blocks.items():
    print(k, df.shape)
X_pareto = blocks["X_pareto"]
X_log = blocks["X_log"]

X_proc, steps = preprocess_feature_matrix(X_raw)
print("Klasszikus LC–MS lépések:", " → ".join(steps))
X_proc.head()


## 3. PCA és explroatív adatstruktúra


In [ ]:
from src.metabolomics_multivariate import fit_pca
from src.metabolomics_plotting import plot_pca_multipanel, plot_pca_scores

pca_out = fit_pca(X_pareto, n_components=5, standardize=False)
scores = pca_out["scores"]
var = pca_out["explained_variance_ratio"]
print("Magyarázott variancia (első 5 PC):", [round(float(v), 4) for v in var])

fig = plot_pca_multipanel(pca_out, meta, hue_col="class_label")
fig.savefig("pca_multipanel_mtbls28.png", dpi=150, bbox_inches="tight")

fig2 = plot_pca_scores(
    scores, meta,
    pc_x="PC1", pc_y="PC2",
    hue_col="class_label",
    explained=var,
    title="PCA score (Pareto-skálázott log1p, MTBLS28)",
)
fig2.savefig("pca_scores_mtbls28.png", dpi=150, bbox_inches="tight")


## 4. Exploratív feature selection és biomarkerjelölt-keresés
### 4.1 Univariáns elemzés, FDR, volcano plot

In [ ]:
# MTBLS28: minden minta egy vizeletes profil; csoport = class_label
metab = meta

from src.metabolomics_univariate import differential_analysis
from src.metabolomics_plotting import plot_volcano, plot_volcano_categorized

diff = differential_analysis(
    X_log,
    metab["class_label"],
    group_a="Lung cancer",
    group_b="Control",
    include_cohen_d=True,
)
print(diff.head(15))
fig = plot_volcano(diff, alpha=0.05, fc_thresh=0.5)
fig.savefig("volcano_lung_cancer_vs_control_mtbls28.png", dpi=150, bbox_inches="tight")
figc = plot_volcano_categorized(
    diff,
    group_up="Lung cancer",
    group_down="Control",
    alpha=0.05,
    fc_thresh=0.5,
)
figc.savefig("volcano_categorized_mtbls28.png", dpi=150, bbox_inches="tight")



### 4.2 Top feature heatmap


In [ ]:
from src.metabolomics_plotting import plot_top_features_heatmap

top_n = 25
top_feats = diff.nsmallest(top_n, "padj")["feature"].tolist()
fig = plot_top_features_heatmap(
    X_pareto,
    top_feats,
    metab,
    group_col="class_label",
    max_samples=50,
)
fig.savefig("heatmap_top_features_mtbls28.png", dpi=150, bbox_inches="tight")


### 4.3 PLS-DA, VIP és S-plot


In [ ]:
from src.metabolomics_multivariate import fit_plsda_multiclass
from src.metabolomics_plotting import plot_s_plot_lv1
import matplotlib.pyplot as plt

plsda = fit_plsda_multiclass(
    X_pareto,
    metab["class_label"],
    n_components=3,
    scale=True,
)
s_pls = plsda["scores"]
vip = plsda["vip"]
print(vip.nlargest(10))

fig, ax = plt.subplots(figsize=(7, 5))
for lab in metab["class_label"].dropna().unique():
    m = metab["class_label"] == lab
    ax.scatter(s_pls.loc[m, "LV1"], s_pls.loc[m, "LV2"], label=str(lab), s=45, alpha=0.85, edgecolors="white", linewidths=0.3)
ax.set_xlabel("LV1")
ax.set_ylabel("LV2")
ax.set_title("PLS-DA score (MTBLS28: class_label)")
ax.legend(title="class_label")
ax.axhline(0, color="gray", lw=0.4)
ax.axvline(0, color="gray", lw=0.4)
fig.tight_layout()
fig.savefig("plsda_scores_mtbls28.png", dpi=150, bbox_inches="tight")

w1 = plsda["model"].pls.x_weights_[:, 0]
figs = plot_s_plot_lv1(X_pareto, metab["class_label"], "Lung cancer", w1)
figs.savefig("splot_lv1_mtbls28.png", dpi=150, bbox_inches="tight")


### 4.4 Random Forest feature importance

In [ ]:
from src.metabolomics_ml import random_forest_feature_importance

imp = random_forest_feature_importance(
    X_pareto,
    metab["class_label"],
    "Lung cancer",
    n_estimators=300,
)
print(imp.head(20))


## 5. Validált ML pipeline-ok: preprocessing, feature selection és modell-összehasonlítás
### 5.1 Feature selection stratégiák
Cél: **Lung cancer vs Control** klasszifikáció validált, leakage-mentes pipeline-okkal.

Bemenet: `X_raw`, `y = meta["class_label"]`.
Pozitív osztály: `Lung cancer`, negatív osztály: `Control`.

A validált ML-ben elsődlegesen **nem** az előre számolt `X_pareto`-ból indulunk, hanem `X_raw`-ból.

### 5.2 Train / validation / test split
- 70% train
- 15% validation
- 15% test

### 5.3 Leakage-mentes preprocessing (pipeline-on belül)
- missing value imputáció (median)
- mintánkénti normalizálás
- `log1p` transzformáció
- alacsony varianciájú feature-ök kiszűrése
- skálázás

`fit`: csak train adaton, `transform`: validation/test adaton.

### 5.4 Pipeline-on belüli feature selection
- `SelectKBest(f_classif)` (több `k` értékkel)
- opcionálisan L1-logisztikus regresszió alapú szelekció

### 5.5 Modellek
- Lasso (L1 logistic regression)
- Ridge (L2 logistic regression)
- Random Forest
- XGBoost (ha elérhető)
- Neural Network (TensorFlow MLP)

### 5.6 Modell-összehasonlítás
Validációs ROC-AUC alapján modellenként legjobb konfiguráció kiválasztása, majd egyszeri test értékelés.

### 5.7 ROC-görbék és confusion matrixok
Publikációs jellegű összehasonlító ábrák mentése.

### 5.8 ML-alapú top feature-ök
Integrált feature-tábla: univariáns, PLS-DA VIP, exploratív RF és validált ML feature-információk.

In [ ]:
from src.metabolomics_ml_benchmark import run_ml_benchmark

ml_results = run_ml_benchmark(
    X_raw,
    metab["class_label"],
    output_dir="outputs/ml_benchmark_mtbls28",
    positive_label="Lung cancer",
    random_state=42,
    k_values=(50, 100, 200, 500),
    refit_on_train_valid=True,
    selector_method="kbest",
    verbose=True,
)

metrics = ml_results["metrics"]
predictions = ml_results["predictions"]
selected_features = ml_results["selected_features"]

metrics.sort_values(["model", "dataset"])

### 5.6 Modell-összehasonlítás és vizualizáció

In [ ]:
import numpy as np

from src.metabolomics_ml_benchmark import (
    plot_metric_comparison,
    plot_roc_curves,
    plot_top_features,
)

# Modellösszehasonlítás
plot_metric_comparison(metrics, metric="roc_auc")
plot_metric_comparison(metrics, metric="f1")
plot_roc_curves(predictions)

# Top feature ábrák
plot_top_features(selected_features, top_n=20)

# Integrált feature tábla az exploratív és validált eredményekből
univar_rank = diff[["feature", "padj"]].copy().sort_values("padj").reset_index(drop=True)
univar_rank["univariate_rank"] = np.arange(1, len(univar_rank) + 1)
univar_rank = univar_rank[["feature", "univariate_rank"]]

vip_rank = vip.sort_values(ascending=False).rename("vip").reset_index()
vip_rank.columns = ["feature", "vip"]
vip_rank["plsda_vip_rank"] = np.arange(1, len(vip_rank) + 1)
vip_rank = vip_rank[["feature", "plsda_vip_rank"]]

rf_expl_rank = imp.sort_values(ascending=False).rename("rf_expl_importance").reset_index()
rf_expl_rank.columns = ["feature", "rf_expl_importance"]
rf_expl_rank["rf_expl_rank"] = np.arange(1, len(rf_expl_rank) + 1)
rf_expl_rank = rf_expl_rank[["feature", "rf_expl_rank"]]

sel = selected_features.copy()
sel["lasso_selected"] = np.where(sel["model"] == "lasso", 1, 0)
sel_lasso = sel.groupby("feature", as_index=False)["lasso_selected"].max()

rf_ml = sel[sel["model"] == "random_forest"][["feature", "importance"]].rename(
    columns={"importance": "rf_ml_importance"}
)
xgb_ml = sel[sel["model"] == "xgboost"][["feature", "importance"]].rename(
    columns={"importance": "xgboost_importance"}
)

feature_summary = (
    univar_rank
    .merge(vip_rank, on="feature", how="outer")
    .merge(rf_expl_rank, on="feature", how="outer")
    .merge(sel_lasso, on="feature", how="left")
    .merge(rf_ml, on="feature", how="left")
    .merge(xgb_ml, on="feature", how="left")
)
feature_summary["lasso_selected"] = feature_summary["lasso_selected"].fillna(0).astype(int)

feature_summary = feature_summary.sort_values(
    ["univariate_rank", "plsda_vip_rank", "rf_expl_rank"],
    na_position="last",
).reset_index(drop=True)

feature_summary.head(30)

### 5.8 ML-alapú top feature-ök
A `feature_summary` tábla kombinálja az univariáns, PLS-DA, RF (exploratív) és validált ML feature-információkat.